# Underfitting and Overfitting with Polynomial Regression

This notebook demonstrates how model complexity affects generalization by fitting polynomial regression models of different degrees to a small synthetic dataset.

## Learning Objectives

After completing this notebook, students should be able to:

1. distinguish between underfitting, appropriate fitting, and overfitting;
2. explain how polynomial degree controls model complexity;
3. identify high-bias and high-variance behavior;
4. use cross-validation to estimate prediction error;
5. interpret mean squared error across models;
6. explain why training fit alone is insufficient for model selection.

## Experimental Design

The underlying relationship is defined by:

\[
f(x) = \cos(1.5\pi x)
\]

Thirty input values are sampled from the interval \([0,1]\). Random Gaussian noise is added to the true function values to simulate imperfect observations.

Three polynomial models are evaluated:

- degree 1;
- degree 4;
- degree 15.

Each model is constructed as a scikit-learn pipeline containing:

1. `PolynomialFeatures`, which transforms the original input into polynomial terms;
2. `LinearRegression`, which estimates the coefficients of those terms.

## Model Complexity

Increasing the polynomial degree increases the flexibility of the model.

- A low-degree model may be too simple to represent the true relationship.
- A moderate-degree model may capture the main pattern while remaining stable.
- A high-degree model may fit noise and fluctuate strongly between observations.

This illustrates the bias-variance trade-off:

- simpler models usually have higher bias and lower variance;
- more complex models usually have lower bias and higher variance.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score


def true_fun(X):
    return np.cos(1.5 * np.pi * X)


np.random.seed(0)

n_samples = 30
degrees = [1, 4, 15]

X = np.sort(np.random.rand(n_samples))
y = true_fun(X) + np.random.randn(n_samples) * 0.1

plt.figure(figsize=(14, 5))
for i in range(len(degrees)):
    ax = plt.subplot(1, len(degrees), i + 1)
    plt.setp(ax, xticks=(), yticks=())

    polynomial_features = PolynomialFeatures(degree=degrees[i], include_bias=False)
    linear_regression = LinearRegression()
    pipeline = Pipeline(
        [
            ("polynomial_features", polynomial_features),
            ("linear_regression", linear_regression),
        ]
    )
    pipeline.fit(X[:, np.newaxis], y)

    # Evaluate the models using crossvalidation
    scores = cross_val_score(
        pipeline, X[:, np.newaxis], y, scoring="neg_mean_squared_error", cv=10
    )

    X_test = np.linspace(0, 1, 100)
    plt.plot(X_test, pipeline.predict(X_test[:, np.newaxis]), label="Model")
    plt.plot(X_test, true_fun(X_test), label="True function")
    plt.scatter(X, y, edgecolor="b", s=20, label="Samples")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.xlim((0, 1))
    plt.ylim((-2, 2))
    plt.legend(loc="best")
    plt.title(
        "Degree {}\nMSE = {:.2e}(+/- {:.2e})".format(
            degrees[i], -scores.mean(), scores.std()
        )
    )
plt.show()

## Result Interpretation

### Degree 1: Underfitting

The degree-1 model produces a straight line.

It cannot represent the nonlinear cosine-shaped relationship, so it performs poorly even on the general structure of the data. This is an example of:

- high bias;
- low model flexibility;
- underfitting.

### Degree 4: Appropriate Complexity

The degree-4 model generally follows the shape of the true function without reacting excessively to individual noisy observations.

It usually provides the best balance between:

- fitting the observed samples;
- approximating the true relationship;
- generalizing to unseen data.

### Degree 15: Overfitting

The degree-15 model is highly flexible and may pass very close to the training observations.

However, it can produce strong oscillations between data points because it begins modeling random noise rather than only the underlying relationship. This is an example of:

- low training bias;
- high variance;
- overfitting;
- unstable predictions.

## Cross-Validation

Each model is evaluated using 10-fold cross-validation:

```python
cross_val_score(
    pipeline,
    X[:, np.newaxis],
    y,
    scoring="neg_mean_squared_error",
    cv=10,
)